# Project 9: Quantum Error Mitigation Benchmarking

## Introduction

Current quantum processors operate in the **Noisy Intermediate-Scale Quantum (NISQ)** era, where qubits suffer from decoherence, gate errors, and measurement noise. While full **quantum error correction (QEC)** requires thousands of physical qubits per logical qubit (using codes such as the surface code), **quantum error mitigation (QEM)** techniques allow us to extract more accurate results from noisy hardware without the overhead of full error correction.

### Error Correction vs. Error Mitigation

| Feature | Error Correction (QEC) | Error Mitigation (QEM) |
|---------|----------------------|----------------------|
| **Qubit overhead** | Very high (1000+ physical per logical) | None or minimal |
| **Circuit overhead** | Syndrome measurements, decoding | Multiple circuit runs, classical post-processing |
| **Accuracy** | Can achieve fault tolerance | Reduces but does not eliminate error |
| **NISQ compatible** | No | Yes |
| **Scalability** | Asymptotically optimal | Sampling cost grows exponentially in general |

This project implements and benchmarks three key error mitigation techniques:
1. **Zero-Noise Extrapolation (ZNE)**
2. **Measurement Error Mitigation**
3. **Probabilistic Error Cancellation (PEC)**

In [ ]:
# --- Imports ---
import pennylane as qml
from pennylane import numpy as pnp
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.linalg import inv
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("PennyLane version:", qml.__version__)
print("NumPy version:", np.__version__)

## Noise Models

### Depolarizing Channel

The single-qubit depolarizing channel with error probability $p$ acts on a density matrix $\rho$ as:

$$\mathcal{E}(\rho) = (1-p)\rho + \frac{p}{3}(X\rho X + Y\rho Y + Z\rho Z)$$

This randomly applies one of the three Pauli errors with equal probability $p/3$, while leaving the state unchanged with probability $1-p$. The depolarizing channel is the most commonly used noise model as it captures the symmetric loss of quantum information.

### Amplitude Damping Channel

The amplitude damping channel models energy dissipation ($T_1$ decay) and is described by Kraus operators:

$$K_0 = \begin{pmatrix} 1 & 0 \\ 0 & \sqrt{1-\gamma} \end{pmatrix}, \quad K_1 = \begin{pmatrix} 0 & \sqrt{\gamma} \\ 0 & 0 \end{pmatrix}$$

where $\gamma$ is the damping parameter. This channel drives the qubit towards the $|0\rangle$ state.

### Bit-Flip (Measurement) Error

Measurement errors flip the classical readout bit with probability $p_m$:

$$P(\text{read } 1 | \text{true } 0) = p_m, \quad P(\text{read } 0 | \text{true } 1) = p_m$$

In [ ]:
# --- Noisy Simulations with default.mixed ---

def create_noisy_ghz(n_qubits, noise_param, noise_type='depolarizing'):
    """
    Create a noisy GHZ state preparation circuit.
    Returns probabilities of all computational basis states.
    """
    dev = qml.device('default.mixed', wires=n_qubits)
    
    @qml.qnode(dev)
    def circuit():
        # GHZ state: H on qubit 0, then cascade of CNOTs
        qml.Hadamard(wires=0)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        
        # Apply noise channel to each qubit
        for i in range(n_qubits):
            if noise_type == 'depolarizing':
                qml.DepolarizingChannel(noise_param, wires=i)
            elif noise_type == 'amplitude_damping':
                qml.AmplitudeDamping(noise_param, wires=i)
        
        return qml.probs(wires=range(n_qubits))
    
    return circuit


def create_random_circuit(n_qubits, depth, noise_param):
    """
    Create a random circuit with depolarizing noise.
    Uses a fixed random seed for reproducibility.
    """
    dev = qml.device('default.mixed', wires=n_qubits)
    rng = np.random.RandomState(123)
    angles = rng.uniform(0, 2 * np.pi, size=(depth, n_qubits, 3))
    
    @qml.qnode(dev)
    def circuit():
        for d in range(depth):
            for q in range(n_qubits):
                qml.RX(angles[d, q, 0], wires=q)
                qml.RY(angles[d, q, 1], wires=q)
                qml.RZ(angles[d, q, 2], wires=q)
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])
            # Apply noise after each layer
            for q in range(n_qubits):
                qml.DepolarizingChannel(noise_param, wires=q)
        return qml.probs(wires=range(n_qubits))
    
    return circuit


# --- Demonstrate noise effects ---
n_qubits = 3
noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# GHZ state under depolarizing noise
print("=== GHZ State under Depolarizing Noise ===")
basis_labels = [format(i, f'0{n_qubits}b') for i in range(2**n_qubits)]

for p in noise_levels:
    circ = create_noisy_ghz(n_qubits, p, 'depolarizing')
    probs = circ()
    axes[0].bar(np.arange(2**n_qubits) + noise_levels.index(p) * 0.15,
                probs, width=0.14, label=f'p={p}', alpha=0.8)
    if p in [0.0, 0.1]:
        print(f"  p={p}: P(000)={probs[0]:.4f}, P(111)={probs[-1]:.4f}")

axes[0].set_xticks(np.arange(2**n_qubits) + 0.3)
axes[0].set_xticklabels(basis_labels)
axes[0].set_xlabel('Basis State')
axes[0].set_ylabel('Probability')
axes[0].set_title('3-Qubit GHZ State: Depolarizing Noise', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Random circuit fidelity degradation
print("\n=== Random Circuit under Depolarizing Noise ===")
ideal_circ = create_random_circuit(n_qubits, depth=3, noise_param=0.0)
ideal_probs = ideal_circ()

fidelities = []
p_range = np.linspace(0, 0.2, 20)
for p in p_range:
    noisy_circ = create_random_circuit(n_qubits, depth=3, noise_param=p)
    noisy_probs = noisy_circ()
    # Classical fidelity (Bhattacharyya coefficient squared)
    fid = np.sum(np.sqrt(ideal_probs * noisy_probs)) ** 2
    fidelities.append(fid)

axes[1].plot(p_range, fidelities, 'b-o', markersize=4, linewidth=2)
axes[1].set_xlabel('Noise Parameter p')
axes[1].set_ylabel('Fidelity with Ideal')
axes[1].set_title('Random Circuit: Fidelity vs Noise', fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Ideal')
axes[1].legend()

plt.tight_layout()
plt.show()

## Zero-Noise Extrapolation (ZNE)

ZNE is based on the idea that if we can controllably **amplify** the noise in a circuit, we can extrapolate back to the **zero-noise** limit.

### Method

1. Run the circuit at multiple noise scale factors $\lambda = 1, 2, 3, \ldots$
2. Measure the expectation value $E(\lambda)$ at each noise level
3. Fit a model (e.g., polynomial or exponential) to $E(\lambda)$
4. Extrapolate to $\lambda = 0$ to estimate the noise-free value

### Richardson Extrapolation

For noise levels $\lambda_1, \lambda_2, \ldots, \lambda_n$, the Richardson extrapolation gives:

$$E(0) = \sum_{i=1}^{n} \gamma_i E(\lambda_i)$$

where the coefficients $\gamma_i$ are chosen so that leading error terms cancel. For $\lambda = 1, 2, 3$:

$$E(0) = 3 E(1) - 3 E(2) + E(3)$$

In [ ]:
# --- Zero-Noise Extrapolation (ZNE) Implementation ---

def run_circuit_at_noise(base_noise, scale_factor, n_qubits=3, observable_wire=0):
    """
    Run a GHZ circuit with noise scaled by scale_factor.
    Returns expectation value of PauliZ on specified wire.
    """
    scaled_noise = min(base_noise * scale_factor, 1.0)
    dev = qml.device('default.mixed', wires=n_qubits)
    
    @qml.qnode(dev)
    def circuit():
        qml.Hadamard(wires=0)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(n_qubits):
            qml.DepolarizingChannel(scaled_noise, wires=i)
        return qml.expval(qml.PauliZ(observable_wire))
    
    return circuit()


def richardson_extrapolation(scale_factors, expectation_values):
    """
    Perform Richardson extrapolation to estimate the zero-noise value.
    
    Solves the Vandermonde system for coefficients gamma_i such that:
    sum(gamma_i) = 1 and sum(gamma_i * lambda_i^k) = 0 for k=1,...,n-1
    """
    n = len(scale_factors)
    V = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            V[i, j] = scale_factors[j] ** i
    
    rhs = np.zeros(n)
    rhs[0] = 1.0
    
    gammas = np.linalg.solve(V, rhs)
    extrapolated = np.dot(gammas, expectation_values)
    return extrapolated, gammas


# --- Run ZNE for multiple base noise levels ---
base_noise_levels = [0.01, 0.05, 0.1]
scale_factors = [1, 2, 3]

# Get ideal (noiseless) value
ideal_value = run_circuit_at_noise(0.0, 1, n_qubits=3)
print(f"Ideal (noiseless) expectation value: {ideal_value:.6f}")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, base_p in enumerate(base_noise_levels):
    # Measure at each scale factor
    exp_vals = []
    for lam in scale_factors:
        val = run_circuit_at_noise(base_p, lam)
        exp_vals.append(val)
    
    # Richardson extrapolation
    zne_value, gammas = richardson_extrapolation(scale_factors, exp_vals)
    
    # Linear extrapolation for comparison
    coeffs = np.polyfit(scale_factors, exp_vals, 1)
    linear_value = np.polyval(coeffs, 0)
    
    print(f"\nBase noise p={base_p}:")
    for lam, val in zip(scale_factors, exp_vals):
        print(f"  lambda={lam}: E = {val:.6f}")
    print(f"  Richardson ZNE estimate: {zne_value:.6f}")
    print(f"  Linear extrapolation:    {linear_value:.6f}")
    print(f"  Noisy (no mitigation):   {exp_vals[0]:.6f}")
    print(f"  Ideal:                   {ideal_value:.6f}")
    print(f"  Error reduction: {abs(exp_vals[0]-ideal_value):.6f} -> {abs(zne_value-ideal_value):.6f}")
    
    # Visualization
    lam_plot = np.linspace(0, 3.5, 100)
    poly_coeffs = np.polyfit(scale_factors, exp_vals, len(scale_factors) - 1)
    axes[idx].plot(lam_plot, np.polyval(poly_coeffs, lam_plot), 'b--', alpha=0.5,
                   label='Richardson fit')
    axes[idx].plot(lam_plot, np.polyval(coeffs, lam_plot), 'g--', alpha=0.5,
                   label='Linear fit')
    axes[idx].scatter(scale_factors, exp_vals, color='red', s=100, zorder=5,
                      label='Measured')
    axes[idx].scatter([0], [zne_value], color='blue', s=150, marker='*', zorder=5,
                      label=f'ZNE = {zne_value:.4f}')
    axes[idx].scatter([0], [linear_value], color='green', s=100, marker='D', zorder=5,
                      label=f'Linear = {linear_value:.4f}')
    axes[idx].axhline(y=ideal_value, color='black', linestyle=':', alpha=0.7,
                      label=f'Ideal = {ideal_value:.4f}')
    axes[idx].set_xlabel('Noise Scale Factor $\\lambda$', fontsize=11)
    axes[idx].set_ylabel('$\\langle Z \\rangle$', fontsize=11)
    axes[idx].set_title(f'ZNE: base p = {base_p}', fontsize=12, fontweight='bold')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Measurement Error Mitigation

Measurement errors can be characterized and corrected using a **calibration matrix** approach.

### Method

1. **Calibration**: Prepare each computational basis state $|b\rangle$ and measure to build the confusion matrix $M$, where $M_{ij} = P(\text{measure } i | \text{prepared } j)$.

2. **Correction**: Given noisy measurement distribution $\vec{p}_{\text{noisy}}$, the corrected distribution is:

$$\vec{p}_{\text{corrected}} = M^{-1} \vec{p}_{\text{noisy}}$$

3. **Clipping**: Since $M^{-1}$ may produce negative probabilities, we clip and renormalize.

For $n$ qubits, the calibration matrix has size $2^n \times 2^n$, making this approach practical for small systems.

In [ ]:
# --- Measurement Error Mitigation ---

def build_calibration_matrix(n_qubits, meas_error_prob):
    """
    Build the measurement calibration matrix M.
    M[i,j] = P(measure state i | true state is j)
    
    Assumes independent bit-flip errors with probability meas_error_prob.
    """
    n_states = 2 ** n_qubits
    M = np.zeros((n_states, n_states))
    
    for true_state in range(n_states):
        true_bits = [(true_state >> q) & 1 for q in range(n_qubits)]
        
        for meas_state in range(n_states):
            meas_bits = [(meas_state >> q) & 1 for q in range(n_qubits)]
            
            prob = 1.0
            for q in range(n_qubits):
                if true_bits[q] == meas_bits[q]:
                    prob *= (1 - meas_error_prob)
                else:
                    prob *= meas_error_prob
            M[meas_state, true_state] = prob
    
    return M


def apply_measurement_noise(ideal_probs, M):
    """Apply measurement error by multiplying with calibration matrix."""
    return M @ ideal_probs


def mitigate_measurement(noisy_probs, M):
    """Apply inverse calibration matrix and clip to valid probabilities."""
    M_inv = inv(M)
    corrected = M_inv @ noisy_probs
    corrected = np.clip(corrected, 0, None)
    corrected /= corrected.sum()
    return corrected


# --- Apply to GHZ state ---
n_qubits = 3
n_states = 2 ** n_qubits
basis_labels = [format(i, f'0{n_qubits}b') for i in range(n_states)]

# Ideal GHZ state: equal superposition of |000> and |111>
dev_ideal = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev_ideal)
def ideal_ghz():
    qml.Hadamard(wires=0)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])
    return qml.probs(wires=range(n_qubits))

ideal_probs = ideal_ghz()
print("Ideal GHZ probabilities:")
for label, p in zip(basis_labels, ideal_probs):
    if p > 0.001:
        print(f"  |{label}> : {p:.4f}")

# Test with different measurement error rates
meas_errors = [0.02, 0.05, 0.1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, p_meas in enumerate(meas_errors):
    M = build_calibration_matrix(n_qubits, p_meas)
    noisy_probs = apply_measurement_noise(np.array(ideal_probs), M)
    corrected_probs = mitigate_measurement(noisy_probs, M)
    
    x = np.arange(n_states)
    width = 0.25
    axes[idx].bar(x - width, ideal_probs, width, label='Ideal', color='green', alpha=0.8)
    axes[idx].bar(x, noisy_probs, width, label='Noisy', color='red', alpha=0.8)
    axes[idx].bar(x + width, corrected_probs, width, label='Corrected', color='blue', alpha=0.8)
    
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(basis_labels, fontsize=8)
    axes[idx].set_xlabel('Basis State')
    axes[idx].set_ylabel('Probability')
    axes[idx].set_title(f'Measurement Mitigation\n(error rate = {p_meas})',
                         fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Compute and print fidelities
    noisy_fid = np.sum(np.sqrt(np.array(ideal_probs) * noisy_probs)) ** 2
    corr_fid = np.sum(np.sqrt(np.array(ideal_probs) * corrected_probs)) ** 2
    print(f"\nMeas error = {p_meas}: Noisy fidelity = {noisy_fid:.4f}, "
          f"Corrected fidelity = {corr_fid:.4f}")

plt.tight_layout()
plt.show()

# Visualize calibration matrix
M_example = build_calibration_matrix(n_qubits, 0.05)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(M_example, cmap='Blues', vmin=0)
ax.set_xticks(range(n_states))
ax.set_yticks(range(n_states))
ax.set_xticklabels(basis_labels, fontsize=8)
ax.set_yticklabels(basis_labels, fontsize=8)
ax.set_xlabel('True State', fontsize=12)
ax.set_ylabel('Measured State', fontsize=12)
ax.set_title('Calibration Matrix M (p_meas=0.05)', fontsize=13, fontweight='bold')
for i in range(n_states):
    for j in range(n_states):
        ax.text(j, i, f'{M_example[i,j]:.3f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Probabilistic Error Cancellation (PEC)

PEC decomposes an ideal (noiseless) gate as a **quasi-probability distribution** over noisy implementable operations:

$$\mathcal{G}_{\text{ideal}} = \sum_i \eta_i \mathcal{O}_i$$

where $\eta_i$ are quasi-probabilities (can be negative) and $\mathcal{O}_i$ are noisy operations.

### Sampling Protocol

1. Decompose each gate: $\mathcal{G} = \gamma \sum_i p_i \text{sign}(\eta_i) \mathcal{O}_i$ where $\gamma = \sum_i |\eta_i|$
2. Sample operation $\mathcal{O}_i$ with probability $|\eta_i|/\gamma$
3. Run the sampled circuit and multiply result by $\text{sign}(\eta_i) \cdot \gamma$
4. Average over many samples to obtain the unbiased estimate

The **sampling overhead** scales as $\gamma^{2n_g}$ where $n_g$ is the number of gates, making PEC practical only for shallow circuits.

In [ ]:
# --- Simplified Probabilistic Error Cancellation (PEC) ---

def pec_single_qubit_depolarizing(noise_param, n_samples=1000):
    """
    Simplified PEC for a single-qubit circuit under depolarizing noise.
    
    For depolarizing channel with parameter p, the inverse channel has
    quasi-probability decomposition:
      eta_I = (1 - p) / (1 - 4p/3)
      eta_X = eta_Y = eta_Z = -(p/3) / (1 - 4p/3)
    
    The sampling overhead is gamma = sum(|eta_i|).
    """
    p = noise_param
    
    # Quasi-probability coefficients for the inverse channel
    denom = 1 - 4*p/3
    eta_I = (1 - p) / denom
    eta_P = -(p/3) / denom  # same for X, Y, Z corrections
    
    # Sampling overhead
    gamma = abs(eta_I) + 3 * abs(eta_P)
    
    # Sampling probabilities and signs
    probs = np.array([abs(eta_I), abs(eta_P), abs(eta_P), abs(eta_P)]) / gamma
    signs = np.array([np.sign(eta_I), np.sign(eta_P), np.sign(eta_P), np.sign(eta_P)])
    
    # Target circuit: RX(pi/4) gate, measure <Z>
    dev_ideal = qml.device('default.qubit', wires=1)
    @qml.qnode(dev_ideal)
    def ideal_circuit():
        qml.RX(np.pi/4, wires=0)
        return qml.expval(qml.PauliZ(0))
    
    ideal_val = ideal_circuit()
    
    # Noisy value without correction
    dev_noisy = qml.device('default.mixed', wires=1)
    @qml.qnode(dev_noisy)
    def noisy_circuit():
        qml.RX(np.pi/4, wires=0)
        qml.DepolarizingChannel(p, wires=0)
        return qml.expval(qml.PauliZ(0))
    
    noisy_val = noisy_circuit()
    
    # PEC: Monte Carlo sampling over quasi-probability decomposition
    pauli_ops = ['I', 'X', 'Y', 'Z']
    pec_estimates = []
    
    for _ in range(n_samples):
        # Sample which correction Pauli to apply
        idx = np.random.choice(4, p=probs)
        sign = signs[idx]
        
        # Run noisy circuit with the sampled correction gate
        @qml.qnode(dev_noisy)
        def sampled_circuit():
            qml.RX(np.pi/4, wires=0)
            qml.DepolarizingChannel(p, wires=0)
            if pauli_ops[idx] == 'X':
                qml.PauliX(wires=0)
            elif pauli_ops[idx] == 'Y':
                qml.PauliY(wires=0)
            elif pauli_ops[idx] == 'Z':
                qml.PauliZ(wires=0)
            return qml.expval(qml.PauliZ(0))
        
        result = sampled_circuit()
        pec_estimates.append(sign * gamma * result)
    
    pec_val = np.mean(pec_estimates)
    pec_std = np.std(pec_estimates) / np.sqrt(n_samples)
    
    return ideal_val, noisy_val, pec_val, pec_std, gamma


# --- Run PEC for different noise levels ---
noise_params = [0.01, 0.05, 0.1, 0.15, 0.2]
results_pec = []

print("=== Probabilistic Error Cancellation Results ===")
print(f"{'Noise p':>8} {'Ideal':>8} {'Noisy':>8} {'PEC':>8} {'PEC Std':>8} {'Gamma':>8}")
print("-" * 56)

for p in noise_params:
    ideal_v, noisy_v, pec_v, pec_s, gam = pec_single_qubit_depolarizing(p, n_samples=2000)
    results_pec.append((p, ideal_v, noisy_v, pec_v, pec_s, gam))
    print(f"{p:8.3f} {ideal_v:8.4f} {noisy_v:8.4f} {pec_v:8.4f} {pec_s:8.4f} {gam:8.3f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ps = [r[0] for r in results_pec]
ideals = [r[1] for r in results_pec]
noisys = [r[2] for r in results_pec]
pecs = [r[3] for r in results_pec]
pec_stds = [r[4] for r in results_pec]
gammas = [r[5] for r in results_pec]

axes[0].axhline(y=ideals[0], color='green', linestyle='--', linewidth=2, label='Ideal')
axes[0].plot(ps, noisys, 'r-o', linewidth=2, label='Noisy (unmitigated)')
axes[0].errorbar(ps, pecs, yerr=pec_stds, fmt='b-s', linewidth=2,
                  capsize=4, label='PEC mitigated')
axes[0].set_xlabel('Noise Parameter p', fontsize=12)
axes[0].set_ylabel('$\\langle Z \\rangle$', fontsize=12)
axes[0].set_title('PEC: Expectation Value Recovery', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(ps, gammas, 'k-o', linewidth=2, markersize=8)
axes[1].set_xlabel('Noise Parameter p', fontsize=12)
axes[1].set_ylabel('Sampling Overhead $\\gamma$', fontsize=12)
axes[1].set_title('PEC: Sampling Overhead', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Comprehensive Benchmark

We now systematically compare all three error mitigation techniques across multiple noise levels on the same benchmark circuit, evaluating:

1. **Accuracy**: How close is the mitigated result to the ideal?
2. **Cost**: How many extra circuit executions are needed?
3. **Reliability**: How consistent are the results?

In [ ]:
# --- Comprehensive Benchmark: All Techniques at p = 0.01, 0.05, 0.1 ---

benchmark_noise = [0.01, 0.05, 0.1]
n_qubits_bench = 3


def benchmark_circuit(noise_param, noise_scale=1):
    """Run GHZ benchmark circuit with scaled noise, return <Z_0>."""
    scaled_p = min(noise_param * noise_scale, 0.99)
    dev = qml.device('default.mixed', wires=n_qubits_bench)
    
    @qml.qnode(dev)
    def circ():
        qml.Hadamard(wires=0)
        for i in range(n_qubits_bench - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(n_qubits_bench):
            qml.DepolarizingChannel(scaled_p, wires=i)
        return qml.expval(qml.PauliZ(0))
    
    return circ()


def benchmark_probs(noise_param, meas_error=None):
    """Get probability distribution for measurement mitigation benchmark."""
    dev = qml.device('default.mixed', wires=n_qubits_bench)
    
    @qml.qnode(dev)
    def circ():
        qml.Hadamard(wires=0)
        for i in range(n_qubits_bench - 1):
            qml.CNOT(wires=[i, i + 1])
        for i in range(n_qubits_bench):
            qml.DepolarizingChannel(noise_param, wires=i)
        return qml.probs(wires=range(n_qubits_bench))
    
    probs = np.array(circ())
    
    if meas_error is not None:
        M = build_calibration_matrix(n_qubits_bench, meas_error)
        probs = apply_measurement_noise(probs, M)
    
    return probs


# Ideal reference values
ideal_expval = benchmark_circuit(0.0)
ideal_probs_bench = benchmark_probs(0.0)

print(f"Ideal <Z_0> = {ideal_expval:.6f}")
print("=" * 80)

results_table = []

for p in benchmark_noise:
    print(f"\n--- Noise level p = {p} ---")
    row = {'noise': p}
    
    # 1. No mitigation
    noisy_val = benchmark_circuit(p)
    row['noisy'] = noisy_val
    row['noisy_error'] = abs(noisy_val - ideal_expval)
    print(f"  No mitigation:    <Z> = {noisy_val:.6f}, error = {row['noisy_error']:.6f}")
    
    # 2. ZNE (Richardson with lambda=1,2,3)
    zne_vals = [benchmark_circuit(p, lam) for lam in [1, 2, 3]]
    zne_result, _ = richardson_extrapolation([1, 2, 3], zne_vals)
    row['zne'] = zne_result
    row['zne_error'] = abs(zne_result - ideal_expval)
    print(f"  ZNE:              <Z> = {zne_result:.6f}, error = {row['zne_error']:.6f}")
    
    # 3. Measurement mitigation (with added meas error = p/2)
    meas_err = p / 2
    noisy_probs_with_meas = benchmark_probs(p, meas_error=meas_err)
    M = build_calibration_matrix(n_qubits_bench, meas_err)
    corrected_probs = mitigate_measurement(noisy_probs_with_meas, M)
    # Convert probabilities to <Z_0> expectation value
    mm_expval = 0
    for state_idx in range(2**n_qubits_bench):
        bit0 = (state_idx >> 0) & 1
        mm_expval += corrected_probs[state_idx] * (1 - 2*bit0)
    row['meas_mit'] = mm_expval
    row['meas_mit_error'] = abs(mm_expval - ideal_expval)
    print(f"  Meas. mitigation: <Z> = {mm_expval:.6f}, error = {row['meas_mit_error']:.6f}")
    
    # 4. PEC
    _, _, pec_v, pec_s, gam = pec_single_qubit_depolarizing(p, n_samples=3000)
    row['pec'] = pec_v
    row['pec_error'] = abs(pec_v - ideal_expval)
    row['pec_gamma'] = gam
    print(f"  PEC:              <Z> = {pec_v:.6f}, error = {row['pec_error']:.6f}")
    
    results_table.append(row)

# --- Comparison Bar Charts and Fidelity Table ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Error comparison (log scale)
x = np.arange(len(benchmark_noise))
width = 0.18
methods = ['noisy_error', 'zne_error', 'meas_mit_error', 'pec_error']
labels = ['No Mitigation', 'ZNE', 'Meas. Mitigation', 'PEC']
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']

for i, (method, label, color) in enumerate(zip(methods, labels, colors)):
    vals = [r[method] for r in results_table]
    axes[0].bar(x + i * width, vals, width, label=label, color=color,
                edgecolor='black', alpha=0.85)

axes[0].set_xticks(x + 1.5 * width)
axes[0].set_xticklabels([f'p={p}' for p in benchmark_noise], fontsize=11)
axes[0].set_ylabel('Absolute Error $|E_{mitigated} - E_{ideal}|$', fontsize=11)
axes[0].set_title('Error Comparison Across Techniques', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_yscale('log')

# Results table
axes[1].axis('off')
table_data = []
for r in results_table:
    table_data.append([
        f"{r['noise']}",
        f"{r['noisy']:.4f}",
        f"{r['zne']:.4f}",
        f"{r['meas_mit']:.4f}",
        f"{r['pec']:.4f}",
        f"{ideal_expval:.4f}"
    ])

col_labels = ['Noise p', 'Noisy', 'ZNE', 'Meas. Mit.', 'PEC', 'Ideal']
table = axes[1].table(cellText=table_data, colLabels=col_labels,
                       loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
axes[1].set_title('Expectation Value Comparison Table',
                   fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# --- Cost Analysis: Circuit Overhead and Accuracy vs Cost Tradeoff ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1. Circuit overhead per technique ---
techniques = ['None', 'ZNE\n(3-pt)', 'Meas.\nMit.', 'PEC\n(2000 samples)']
overheads = [
    1,                          # No mitigation: 1 circuit
    3,                          # ZNE: 3 scale factors
    2**n_qubits_bench + 1,      # Meas mit: 2^n calibration + 1 actual
    2000                        # PEC: many quasi-probability samples
]

bars = axes[0].bar(techniques, overheads,
                    color=['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6'],
                    edgecolor='black', alpha=0.85)
for bar, val in zip(bars, overheads):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Number of Circuit Executions', fontsize=11)
axes[0].set_title('Circuit Overhead by Technique', fontsize=13, fontweight='bold')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3, axis='y')

# --- 2. PEC overhead scaling with noise and circuit depth ---
noise_range = np.linspace(0.001, 0.25, 50)
n_gates_list = [1, 5, 10, 20]

for n_gates in n_gates_list:
    gamma_vals = []
    for p in noise_range:
        denom = 1 - 4*p/3
        if denom > 0:
            gamma_single = (1 + 2*p/3) / denom
            gamma_total = gamma_single ** n_gates
        else:
            gamma_total = np.inf
        gamma_vals.append(gamma_total)
    
    axes[1].semilogy(noise_range, gamma_vals, linewidth=2, label=f'{n_gates} gates')

axes[1].set_xlabel('Noise Parameter p', fontsize=11)
axes[1].set_ylabel('Sampling Overhead $\\gamma^{n_g}$', fontsize=11)
axes[1].set_title('PEC Overhead Scaling', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(1, 1e10)

# --- 3. Accuracy vs Cost tradeoff at p=0.05 ---
p_tradeoff = 0.05

# ZNE with different numbers of extrapolation points
zne_costs = []
zne_errors = []
for n_pts in [2, 3, 4, 5]:
    lambdas = list(range(1, n_pts + 1))
    vals = [benchmark_circuit(p_tradeoff, lam) for lam in lambdas]
    zne_est, _ = richardson_extrapolation(lambdas, vals)
    zne_costs.append(n_pts)
    zne_errors.append(abs(zne_est - ideal_expval))

# PEC with different sample counts
pec_costs = []
pec_errors = []
for n_samp in [100, 500, 1000, 2000, 5000]:
    _, _, pec_v, _, _ = pec_single_qubit_depolarizing(p_tradeoff, n_samples=n_samp)
    pec_costs.append(n_samp)
    pec_errors.append(abs(pec_v - ideal_expval))

axes[2].loglog(zne_costs, zne_errors, 'b-o', linewidth=2, markersize=8, label='ZNE')
axes[2].loglog(pec_costs, pec_errors, 'm-s', linewidth=2, markersize=8, label='PEC')
axes[2].axhline(y=abs(benchmark_circuit(p_tradeoff) - ideal_expval),
                color='red', linestyle='--', label='No mitigation')
axes[2].set_xlabel('Circuit Executions (Cost)', fontsize=11)
axes[2].set_ylabel('Absolute Error', fontsize=11)
axes[2].set_title(f'Accuracy vs Cost (p={p_tradeoff})', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Summary of Circuit Costs ===")
print(f"{'Technique':<25} {'Circuits':<15} {'Best For'}")
print("-" * 65)
print(f"{'No mitigation':<25} {'1':<15} {'Baseline'}")
print(f"{'ZNE (3-point)':<25} {'3':<15} {'Low overhead, moderate improvement'}")
print(f"{'Measurement mitigation':<25} {f'{2**n_qubits_bench+1}':<15} {'Correcting readout errors'}")
print(f"{'PEC (2000 samples)':<25} {'2000':<15} {'Best accuracy, high cost'}")

## Conclusion

### Summary of Findings

| Technique | Accuracy | Cost | Scalability | Best Use Case |
|-----------|----------|------|------------|---------------|
| **ZNE** | Good | Low (3-5x) | Good | General expectation values |
| **Measurement Mitigation** | Excellent for readout errors | Moderate ($2^n$ calibration) | Limited by $2^n$ matrix | Systems dominated by readout errors |
| **PEC** | Best (unbiased) | Very high ($\gamma^{2n_g}$) | Poor for deep circuits | Shallow circuits, high precision |

### Recommendations

1. **Always apply measurement error mitigation** when possible -- it corrects a significant and well-characterized error source with modest overhead.

2. **Use ZNE as a default** for mitigating gate errors. The Richardson extrapolation with 3 noise scale factors provides a good accuracy-cost tradeoff.

3. **Reserve PEC for high-precision tasks** on shallow circuits where the exponential sampling cost is acceptable.

4. **Combine techniques**: Measurement mitigation can be combined with ZNE or PEC for the best results, as they address different error sources.

5. **Monitor noise levels**: All techniques degrade as noise increases. For $p > 0.1$, even mitigated results may have significant residual error.

### References

1. Temme, K., Bravyi, S., & Gambetta, J.M. "Error Mitigation for Short-Depth Quantum Circuits." *Physical Review Letters* 119, 180509 (2017).
2. Li, Y. & Benjamin, S.C. "Efficient Variational Quantum Simulator Incorporating Active Error Minimization." *Physical Review X* 7, 021050 (2017).
3. Kandala, A., et al. "Error mitigation extends the computational reach of a noisy quantum processor." *Nature* 567, 491-495 (2019).
4. van den Berg, E., Minev, Z.K., & Temme, K. "Model-free readout-error mitigation for quantum expectation values." *Physical Review A* 105, 032620 (2022).
5. Cai, Z., et al. "Quantum Error Mitigation." *Reviews of Modern Physics* 95, 045005 (2023).
6. Qiskit Textbook: Error Mitigation. https://qiskit.org/textbook/